In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('smart-mcq-solver-challenge')

print("Path to competition files:", path)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier

from scipy.sparse import hstack

In [ ]:
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

train.head()

In [ ]:
print(train.columns)
print(test.columns)

In [ ]:
def combine(df):
    return (
        "Question: " + df["prompt"].fillna("") +
        " A: " + df["A"].fillna("") +
        " B: " + df["B"].fillna("") +
        " C: " + df["C"].fillna("") +
        " D: " + df["D"].fillna("") +
        " E: " + df["E"].fillna("")
    )

train_text = combine(train)
test_text = combine(test)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1,2),
)

X_train = tfidf.fit_transform(train_text)
X_test = tfidf.transform(test_text)

In [ ]:
le = LabelEncoder()

y = le.fit_transform(train["answer"])

In [ ]:
model = LGBMClassifier(
    objective="multiclass",
    n_estimators=270,
    learning_rate=0.05,
    num_leaves=63,
    random_state=42
)

model.fit(X_train, y)

In [ ]:
probs = model.predict_proba(X_test)

probs.shape

In [ ]:
labels = le.classes_

predictions = []

for row in probs:
    idx = np.argsort(row)[::-1][:3]
    predictions.append(" ".join(labels[idx]))

predictions[:5]

In [ ]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

submission.head()